# Prepare and Merge TACO + AquaTrash into YOLO-ready Dataset

This standalone notebook mirrors scripts/prepare_datasets_to_merge.py and scripts/generated_yolo_ready_dataset.py without command-line arguments. Configure the variables in the next cell, then run all cells.

Set YOLO_TASK = "seg" to prepare/train the 8-class YOLO segmentation dataset with yolo26n-seg.pt. Set YOLO_TASK = "detect" to prepare/train a one-class trash YOLO detection dataset with yolo26m.pt.

Expected input layout:

- TACO does not need to be attached as a Kaggle dataset. The notebook downloads annotations.json from TACO_JSON_URL, then downloads the referenced images into TACO_ROOT_DIR.
- Segmentation mode uses AQUATRASH_ROOT_DIR / annotations.json and AquaTrash images under AQUATRASH_ROOT_DIR / Images / image.jpg.
- Detection mode uses TACO COCO bbox values and AquaTrash raw boxes from AQUATRASH_ROOT_DIR / annotations.csv with images under AQUATRASH_ROOT_DIR / Images / image.jpg.

Outputs are written to NORMALIZED_DIR and one merged YOLO-ready dataset at YOLO_READY_DIR. The later cells build dataset.yaml, normalize labels to YOLO zero-based class IDs, tune/train YOLO with Optuna + MLflow, and evaluate the best weights.


In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlretrieve
import csv
import json
import os
import random
import shutil

IS_KAGGLE = Path("/kaggle").exists()
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")
DATA_DIR = WORKING_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

# Toggle: "seg" builds 8-class segmentation labels, "detect" builds one-class bbox labels named "trash".
YOLO_TASK = os.getenv("YOLO_TASK", "seg").lower().strip()
if YOLO_TASK not in {"seg", "detect"}:
    raise ValueError("YOLO_TASK must be either 'seg' or 'detect'.")

# Edit these paths to match your attached Kaggle AquaTrash dataset.
DATASETS = ["taco", "aquatrash"]
TACO_JSON_FILE = "annotations.json"
TACO_JSON_URL = "https://huggingface.co/datasets/karimaouaouda/taco/resolve/main/annotations.json"
TACO_ROOT_DIR = RAW_DIR / "taco"
TACO_ANNOTATIONS_PATH = TACO_ROOT_DIR / TACO_JSON_FILE
FORCE_TACO_DOWNLOAD = False
TACO_DOWNLOAD_PROGRESS_EVERY = 100

DEFAULT_AQUATRASH_ROOT = Path("/kaggle/input/aquatrash") if IS_KAGGLE else RAW_DIR / "aquatrash"
if not IS_KAGGLE and not DEFAULT_AQUATRASH_ROOT.exists() and Path("../data/raw/aquatrash").exists():
    DEFAULT_AQUATRASH_ROOT = Path("../data/raw/aquatrash")
AQUATRASH_ROOT_DIR = Path(os.getenv("AQUATRASH_ROOT_DIR", str(DEFAULT_AQUATRASH_ROOT)))
AQUATRASH_ANNOTATIONS_PATH = AQUATRASH_ROOT_DIR / "annotations.json"
AQUATRASH_BOXES_CSV_PATH = AQUATRASH_ROOT_DIR / "annotations.csv"

NORMALIZED_DIR = DATA_DIR / "normalized"
YOLO_READY_DIR = DATA_DIR / "yolo_ready"

TRAIN_RATE = 0.7
VAL_RATE = 0.1
TEST_RATE = 0.2

VISUALIZE_AFTER_BUILD = True
VISUALIZE_SPLIT = "train"
VISUALIZE_COUNT = 5

# Training controls. Set RUN_TRAINING = False to only prepare the dataset and YAML.
INSTALL_MISSING_PACKAGES = True
RUN_TRAINING = True
LABEL_ID_MODE = "auto"

DEFAULT_YOLO_MODEL = "yolo26m.pt" if YOLO_TASK == "detect" else "yolo26n-seg.pt"
YOLO_MODEL = os.getenv("YOLO_MODEL", DEFAULT_YOLO_MODEL)
EPOCHS = int(os.getenv("YOLO_EPOCHS", "50"))
TUNE_EPOCHS = int(os.getenv("YOLO_TUNE_EPOCHS", "12"))
OPTUNA_TRIALS = int(os.getenv("YOLO_OPTUNA_TRIALS", "10"))
SKIP_OPTUNA = os.getenv("SKIP_OPTUNA", "0") == "1"
OPTUNA_DIRECTION = "maximize"
OPTUNA_METRIC = os.getenv("YOLO_OPTUNA_METRIC", "box_map50_95" if YOLO_TASK == "detect" else "mask_map50_95")

BATCH = int(os.getenv("YOLO_BATCH", "16"))
IMGSZ = int(os.getenv("YOLO_IMGSZ", "640"))
PATIENCE = int(os.getenv("YOLO_PATIENCE", "20"))
DEVICE = os.getenv("YOLO_DEVICE", "0")
OPTIMIZER = "AdamW"
LR0 = 0.001
LRF = 0.01
WEIGHT_DECAY = 0.0005
AUGMENT = True
MOSAIC = 1.0
MIXUP = 0.1
COPY_PASTE = 0.3 if YOLO_TASK == "seg" else 0.0

RUN_NAME = os.getenv("RUN_NAME", "yolo26m-detect-yolo-ready" if YOLO_TASK == "detect" else "yolo26n-seg-yolo-ready")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "waste-detect-yolo-ready" if YOLO_TASK == "detect" else "waste-seg-yolo-ready")
RUNS_DIR = WORKING_DIR / "runs" / "train"
ARTIFACTS_DIR = WORKING_DIR / "artifacts" / "exports" / f"yolo-ready-{YOLO_TASK}"
MLRUNS_DIR = WORKING_DIR / "mlruns"
DATASET_YAML_PATH = YOLO_READY_DIR / "dataset.yaml"

print("IS_KAGGLE:", IS_KAGGLE)
print("WORKING_DIR:", WORKING_DIR)
print("YOLO_TASK:", YOLO_TASK)
print("TACO_JSON_URL:", TACO_JSON_URL)
print("TACO_ROOT_DIR:", TACO_ROOT_DIR)
print("TACO_ANNOTATIONS_PATH:", TACO_ANNOTATIONS_PATH)
print("AQUATRASH_ROOT_DIR:", AQUATRASH_ROOT_DIR)
print("AQUATRASH_ANNOTATIONS_PATH:", AQUATRASH_ANNOTATIONS_PATH)
print("AQUATRASH_BOXES_CSV_PATH:", AQUATRASH_BOXES_CSV_PATH)
print("NORMALIZED_DIR:", NORMALIZED_DIR)
print("YOLO_READY_DIR:", YOLO_READY_DIR)
print("DATASET_YAML_PATH:", DATASET_YAML_PATH)
print("RUN_TRAINING:", RUN_TRAINING)
print("YOLO_MODEL:", YOLO_MODEL)
print("OPTUNA_METRIC:", OPTUNA_METRIC)
print("EPOCHS:", EPOCHS, "TUNE_EPOCHS:", TUNE_EPOCHS, "OPTUNA_TRIALS:", 0 if SKIP_OPTUNA else OPTUNA_TRIALS)

if IS_KAGGLE and Path("/kaggle/input").exists():
    print("\nAttached Kaggle input roots:")
    for path in sorted(Path("/kaggle/input").iterdir()):
        print(" -", path)


In [ ]:
if INSTALL_MISSING_PACKAGES:
    import importlib.util
    import subprocess
    import sys

    required_packages = [
        ("ultralytics", "ultralytics>=8.0"),
        ("mlflow", "mlflow>=2.14"),
        ("optuna", "optuna>=4.0"),
        ("yaml", "pyyaml>=6.0"),
        ("requests", "requests>=2.0"),
        ("PIL", "pillow>=10.0"),
    ]
    missing_packages = [package for module, package in required_packages if importlib.util.find_spec(module) is None]
    if missing_packages:
        print("Installing missing packages:", missing_packages)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("Training dependencies are already installed.")


In [ ]:
ALLOWED_DATASETS = {"taco", "aquatrash"}
DETECT_CATEGORIES = [{"id": 0, "name": "trash"}]
TACO_TO_AQUATRASH_LABEL = {
    "Food waste": "organic_waste",
    "Other plastic bottle": "plastic_battle",
    "Clear plastic bottle": "plastic_battle",
    "Plastic film": "plastic_bag",
    "Six pack rings": "plastic_bag",
    "Garbage bag": "plastic_bag",
    "Other plastic wrapper": "plastic_bag",
    "Single-use carrier bag": "plastic_bag",
    "Polypropylene bag": "plastic_bag",
    "Crisp packet": "plastic_bag",
    "Plastic bottle cap": "rigid_plastic",
    "Plastic lid": "rigid_plastic",
    "Other plastic": "rigid_plastic",
    "Disposable plastic cup": "rigid_plastic",
    "Foam cup": "rigid_plastic",
    "Other plastic cup": "rigid_plastic",
    "Spread tub": "rigid_plastic",
    "Tupperware": "rigid_plastic",
    "Disposable food container": "rigid_plastic",
    "Foam food container": "rigid_plastic",
    "Other plastic container": "rigid_plastic",
    "Plastic glooves": "rigid_plastic",
    "Plastic utensils": "rigid_plastic",
    "Squeezable tube": "rigid_plastic",
    "Plastic straw": "rigid_plastic",
    "Styrofoam piece": "mixed_waste",
    "Aluminium foil": "metal_can",
    "Aluminium blister pack": "metal_can",
    "Metal bottle cap": "metal_can",
    "Food Can": "metal_can",
    "Aerosol": "metal_can",
    "Drink can": "metal_can",
    "Metal lid": "metal_can",
    "Pop tab": "metal_can",
    "Scrap metal": "metal_can",
    "Glass bottle": "glass",
    "Broken glass": "glass",
    "Glass cup": "glass",
    "Glass jar": "glass",
    "Toilet tube": "paper_cardboard",
    "Other carton": "paper_cardboard",
    "Egg carton": "paper_cardboard",
    "Drink carton": "paper_cardboard",
    "Corrugated carton": "paper_cardboard",
    "Meal carton": "paper_cardboard",
    "Pizza box": "paper_cardboard",
    "Paper cup": "paper_cardboard",
    "Magazine paper": "paper_cardboard",
    "Tissues": "paper_cardboard",
    "Wrapping paper": "paper_cardboard",
    "Normal paper": "paper_cardboard",
    "Paper bag": "paper_cardboard",
    "Plastified paper bag": "paper_cardboard",
    "Paper straw": "paper_cardboard",
    "Battery": "mixed_waste",
    "Carded blister pack": "mixed_waste",
    "Rope & strings": "mixed_waste",
    "Shoe": "mixed_waste",
    "Unlabeled litter": "mixed_waste",
    "Cigarette": "mixed_waste",
}


def validate_dataset(dataset_name: str) -> str:
    if dataset_name not in ALLOWED_DATASETS:
        allowed = ", ".join(sorted(ALLOWED_DATASETS))
        raise ValueError(f"Invalid dataset '{dataset_name}'. Allowed values are: {allowed}.")
    return dataset_name


def load_json(path: Path) -> dict:
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(payload: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle)


def load_aquatrash_categories(annotations_path: Path = AQUATRASH_ANNOTATIONS_PATH) -> list[dict]:
    annotations_path = Path(annotations_path)
    if YOLO_TASK == "detect":
        return DETECT_CATEGORIES
    if not annotations_path.exists():
        raise FileNotFoundError(
            f"AquaTrash annotations file '{annotations_path}' not found. It is required to map TACO classes into the AquaTrash 8-class label space."
        )

    return load_json(annotations_path)["categories"]


def resolve_taco_label(category: dict) -> str:
    name = str(category.get("name", "")).strip()
    supercategory = str(category.get("supercategory", "")).strip()
    lower_name = name.lower()
    lower_supercategory = supercategory.lower()

    if name in TACO_TO_AQUATRASH_LABEL:
        return TACO_TO_AQUATRASH_LABEL[name]

    if lower_supercategory == "bottle":
        return "glass" if "glass" in lower_name else "plastic_battle"
    if lower_supercategory == "bottle cap":
        return "metal_can" if "metal" in lower_name else "rigid_plastic"
    if lower_supercategory in {"paper", "carton", "paper bag"}:
        return "paper_cardboard"
    if lower_supercategory == "plastic bag & wrapper":
        return "plastic_bag"
    if lower_supercategory == "plastic container":
        return "rigid_plastic"
    if lower_supercategory == "cup":
        if lower_name.startswith("glass"):
            return "glass"
        if lower_name.startswith("paper"):
            return "paper_cardboard"
        return "rigid_plastic"
    if lower_supercategory == "lid":
        return "metal_can" if "metal" in lower_name else "rigid_plastic"
    if lower_supercategory == "other plastic":
        return "rigid_plastic"
    if lower_supercategory == "straw":
        return "paper_cardboard" if lower_name.startswith("paper") else "rigid_plastic"
    if lower_supercategory == "food waste":
        return "organic_waste"
    if lower_supercategory in {"battery", "unlabeled litter", "rope & strings", "shoe", "cigarette"}:
        return "mixed_waste"
    if "glass" in lower_name:
        return "glass"
    if "can" in lower_name or "foil" in lower_name or "metal" in lower_name:
        return "metal_can"
    if "paper" in lower_name or "carton" in lower_name or "tissue" in lower_name or "box" in lower_name:
        return "paper_cardboard"
    if (
        "plastic" in lower_name
        or "foam" in lower_name
        or "tupperware" in lower_name
        or "tube" in lower_name
        or "glove" in lower_name
        or "utensil" in lower_name
        or "straw" in lower_name
    ):
        return "rigid_plastic"
    return "mixed_waste"


def clamp_bbox_xywh(bbox: list | tuple | None, width: float, height: float) -> list[float] | None:
    if bbox is None or len(bbox) < 4:
        return None
    x, y, w, h = [float(value) for value in bbox[:4]]
    if w < 0:
        x += w
        w = abs(w)
    if h < 0:
        y += h
        h = abs(h)
    x1 = max(0.0, min(float(width), x))
    y1 = max(0.0, min(float(height), y))
    x2 = max(0.0, min(float(width), x + w))
    y2 = max(0.0, min(float(height), y + h))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2 - x1, y2 - y1]


def bbox_xyxy_to_xywh(x_min, y_min, x_max, y_max, width: float, height: float) -> list[float] | None:
    x1 = max(0.0, min(float(width), float(x_min)))
    y1 = max(0.0, min(float(height), float(y_min)))
    x2 = max(0.0, min(float(width), float(x_max)))
    y2 = max(0.0, min(float(height), float(y_max)))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2 - x1, y2 - y1]


def bbox_from_segmentation(segmentation: list, width: float, height: float) -> list[float] | None:
    if not isinstance(segmentation, list) or len(segmentation) < 6:
        return None
    xs = [float(value) for value in segmentation[0::2]]
    ys = [float(value) for value in segmentation[1::2]]
    if not xs or not ys:
        return None
    return clamp_bbox_xywh([min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)], width, height)


def annotation_bbox(annotation: dict, width: float, height: float) -> list[float] | None:
    bbox = clamp_bbox_xywh(annotation.get("bbox"), width, height)
    if bbox is not None:
        return bbox
    for segmentation in annotation.get("segmentation", []):
        bbox = bbox_from_segmentation(segmentation, width, height)
        if bbox is not None:
            return bbox
    return None


def bbox_area(bbox: list[float] | None) -> float:
    if bbox is None:
        return 0.0
    return float(bbox[2]) * float(bbox[3])


def download_taco_annotations(annotations_path: Path = TACO_ANNOTATIONS_PATH) -> Path:
    annotations_path = Path(annotations_path)
    annotations_path.parent.mkdir(parents=True, exist_ok=True)
    if annotations_path.exists() and not FORCE_TACO_DOWNLOAD:
        print("TACO annotations already exist:", annotations_path)
        return annotations_path

    print("Downloading TACO annotations.json from Hugging Face...")
    urlretrieve(TACO_JSON_URL, annotations_path)
    print("Downloaded TACO annotations to:", annotations_path)
    return annotations_path


def download_taco_images(annotations_path: Path, output_dir: Path, show_progress: bool = True) -> int:
    import requests
    from PIL import Image

    payload = load_json(annotations_path)
    images = payload.get("images", [])
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    downloaded = 0
    existing = 0
    failed = []

    for idx, image in enumerate(images, start=1):
        rel_path = Path(str(image["file_name"]).replace("\\", "/"))
        dst = output_dir / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() and not FORCE_TACO_DOWNLOAD:
            existing += 1
            continue

        urls = [image.get("flickr_url"), image.get("flickr_640_url"), image.get("coco_url")]
        urls = [url for url in urls if url]
        success = False
        last_error = "no_url"
        for url in urls:
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                img = Image.open(BytesIO(response.content)).convert("RGB")
                img.save(dst)
                downloaded += 1
                success = True
                break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"

        if not success:
            failed.append({"file_name": image["file_name"], "urls": urls, "error": last_error})

        if show_progress and TACO_DOWNLOAD_PROGRESS_EVERY and idx % TACO_DOWNLOAD_PROGRESS_EVERY == 0:
            print(f"TACO images checked {idx}/{len(images)} | downloaded={downloaded} existing={existing} failed={len(failed)}")

    summary = {
        "referenced_images": len(images),
        "downloaded": downloaded,
        "existing": existing,
        "failed": len(failed),
        "failed_examples": failed[:20],
        "output_dir": str(output_dir),
    }
    summary_path = DATA_DIR / "taco_download_summary.json"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2)[:2000])
    return len(images) - len(failed)


def ensure_taco_dataset_downloaded(root_dir: Path = TACO_ROOT_DIR, json_file: str = TACO_JSON_FILE) -> Path:
    root_dir = Path(root_dir)
    annotations_path = root_dir / json_file
    annotations_path = download_taco_annotations(annotations_path)
    download_taco_images(annotations_path, root_dir, show_progress=True)
    return annotations_path


def prepare_taco(root_dir: Path, json_file: str = "annotations.json") -> None:
    root_dir = Path(root_dir)
    annotations_path = ensure_taco_dataset_downloaded(root_dir, json_file)
    print(f"Preparing TACO dataset from '{annotations_path.absolute()}' for YOLO_TASK={YOLO_TASK}...")

    dist_dir = NORMALIZED_DIR / "taco"
    dist_dir.mkdir(parents=True, exist_ok=True)
    annotations = load_json(annotations_path)
    target_categories = load_aquatrash_categories() if YOLO_TASK == "seg" else DETECT_CATEGORIES
    target_label_to_id = {category["name"]: category["id"] for category in target_categories}
    taco_categories = {category["id"]: category for category in annotations["categories"]}
    source_images = {image["id"]: image for image in annotations["images"]}
    print(f"Loaded {len(annotations['images'])} images from '{annotations_path}'.")

    prepared_images = []
    available_image_ids = set()
    for image in annotations["images"]:
        file_name = str(image["file_name"]).replace("\\", "/")
        parts = file_name.split("/", 1)
        if len(parts) != 2:
            raise ValueError(f"Expected TACO file_name to look like 'batch_xxx/image.jpg', got '{file_name}'.")

        batch_name, img_name = parts
        new_file_name = f"{batch_name}_{img_name}"
        old_image_path = root_dir / Path(file_name)
        if not old_image_path.exists():
            print(f"Warning: TACO image '{old_image_path}' was not downloaded. This image will be skipped.")
            continue

        prepared_image = dict(image)
        prepared_image["file_name"] = new_file_name
        new_image_path = dist_dir / new_file_name
        new_image_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(old_image_path, new_image_path)
        prepared_images.append(prepared_image)
        available_image_ids.add(image["id"])

    remapped_annotations = []
    skipped_annotations = 0
    for annotation in annotations["annotations"]:
        image_id = annotation["image_id"]
        if image_id not in available_image_ids:
            skipped_annotations += 1
            continue

        source_image = source_images[image_id]
        bbox = annotation_bbox(annotation, width=float(source_image["width"]), height=float(source_image["height"]))
        if YOLO_TASK == "detect":
            if bbox is None:
                skipped_annotations += 1
                continue
            remapped_annotations.append({
                "id": annotation.get("id", len(remapped_annotations) + 1),
                "image_id": image_id,
                "category_id": 0,
                "bbox": bbox,
                "area": bbox_area(bbox),
                "iscrowd": int(annotation.get("iscrowd", 0)),
            })
            continue

        taco_category = taco_categories.get(annotation["category_id"])
        if taco_category is None:
            raise KeyError(f"TACO category id '{annotation['category_id']}' was not found in annotations categories.")

        aquatrash_label = resolve_taco_label(taco_category)
        if aquatrash_label not in target_label_to_id:
            raise KeyError(f"Mapped TACO label '{aquatrash_label}' was not found in AquaTrash categories.")

        remapped_annotation = dict(annotation)
        remapped_annotation["category_id"] = target_label_to_id[aquatrash_label]
        if bbox is not None:
            remapped_annotation["bbox"] = bbox
        remapped_annotations.append(remapped_annotation)

    annotations["images"] = prepared_images
    annotations["annotations"] = remapped_annotations
    annotations["categories"] = target_categories
    print(
        f"Prepared {len(prepared_images)} TACO images and {len(remapped_annotations)} annotations "
        f"({skipped_annotations} annotations skipped for missing images or invalid boxes)."
    )

    write_json(annotations, dist_dir / json_file)


def prepare_aquatrash_segmentation(root_dir: Path, annotations_path: Path) -> None:
    root_dir = Path(root_dir)
    annotations_path = Path(annotations_path)
    if not annotations_path.exists():
        raise FileNotFoundError(
            f"Annotations file '{annotations_path}' not found. Please ensure the AquaTrash segmentation dataset is attached correctly."
        )

    dist_dir = NORMALIZED_DIR / "aquatrash"
    images_dir = root_dir / "Images"
    dist_dir.mkdir(parents=True, exist_ok=True)
    annotations = load_json(annotations_path)

    for image_info in annotations["images"]:
        file_name = image_info["file_name"]
        new_file_name = file_name
        image_info["file_name"] = new_file_name

        old_image_path = images_dir / file_name
        new_image_path = dist_dir / new_file_name
        if not old_image_path.exists():
            raise FileNotFoundError(
                f"Image file '{old_image_path}' not found. Please ensure the AquaTrash dataset is attached correctly."
            )

        new_image_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(old_image_path, new_image_path)

    write_json(annotations, dist_dir / "annotations.json")


def prepare_aquatrash_detection(root_dir: Path, boxes_csv_path: Path) -> None:
    from PIL import Image

    root_dir = Path(root_dir)
    boxes_csv_path = Path(boxes_csv_path)
    if not boxes_csv_path.exists():
        raise FileNotFoundError(
            f"AquaTrash boxes CSV '{boxes_csv_path}' not found. Detection mode expects raw boxes at AQUATRASH_ROOT_DIR / 'annotations.csv'."
        )

    images_dir = root_dir / "Images"
    if not images_dir.exists():
        raise FileNotFoundError(f"AquaTrash Images directory '{images_dir}' was not found.")

    dist_dir = NORMALIZED_DIR / "aquatrash"
    dist_dir.mkdir(parents=True, exist_ok=True)

    rows_by_file = {}
    with boxes_csv_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        required_columns = {"image_name", "x_min", "y_min", "x_max", "y_max", "class_name"}
        missing_columns = required_columns - set(reader.fieldnames or [])
        if missing_columns:
            raise ValueError(f"AquaTrash boxes CSV is missing columns: {sorted(missing_columns)}")
        for row in reader:
            rows_by_file.setdefault(str(row["image_name"]).strip(), []).append(row)

    images = []
    annotations = []
    skipped_rows = 0
    skipped_images = 0
    next_image_id = 1
    for file_name in sorted(rows_by_file):
        old_image_path = images_dir / file_name
        if not old_image_path.exists():
            print(f"Warning: AquaTrash image '{old_image_path}' listed in CSV was not found. It will be skipped.")
            skipped_images += 1
            continue

        with Image.open(old_image_path) as image:
            width, height = image.size

        image_id = next_image_id
        next_image_id += 1
        new_image_path = dist_dir / file_name
        new_image_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(old_image_path, new_image_path)
        images.append({"id": image_id, "width": width, "height": height, "file_name": file_name})

        for row in rows_by_file[file_name]:
            bbox = bbox_xyxy_to_xywh(row["x_min"], row["y_min"], row["x_max"], row["y_max"], width=width, height=height)
            if bbox is None:
                skipped_rows += 1
                continue
            annotations.append({
                "id": len(annotations) + 1,
                "image_id": image_id,
                "category_id": 0,
                "bbox": bbox,
                "area": bbox_area(bbox),
                "iscrowd": 0,
                "source_class_name": str(row.get("class_name", "")).strip(),
            })

    payload = {
        "images": images,
        "annotations": annotations,
        "categories": DETECT_CATEGORIES,
        "info": {
            "source": "AquaTrash",
            "task": "detect",
            "box_source": str(boxes_csv_path),
            "skipped_rows": skipped_rows,
            "skipped_images": skipped_images,
        },
    }
    print(
        f"Prepared {len(images)} AquaTrash images and {len(annotations)} bbox annotations "
        f"from {boxes_csv_path} ({skipped_rows} invalid rows, {skipped_images} missing images skipped)."
    )
    write_json(payload, dist_dir / "annotations.json")


def prepare_aquatrash(root_dir: Path, annotations_path: Path, boxes_csv_path: Path = AQUATRASH_BOXES_CSV_PATH) -> None:
    if YOLO_TASK == "detect":
        prepare_aquatrash_detection(root_dir=root_dir, boxes_csv_path=boxes_csv_path)
    else:
        prepare_aquatrash_segmentation(root_dir=root_dir, annotations_path=annotations_path)


def prepare_dataset(dataset_name: str) -> None:
    validate_dataset(dataset_name)

    if dataset_name == "taco":
        prepare_taco(root_dir=TACO_ROOT_DIR, json_file=TACO_JSON_FILE)
    elif dataset_name == "aquatrash":
        prepare_aquatrash(
            root_dir=AQUATRASH_ROOT_DIR,
            annotations_path=AQUATRASH_ANNOTATIONS_PATH,
            boxes_csv_path=AQUATRASH_BOXES_CSV_PATH,
        )


In [ ]:
SPLITS = ("train", "val", "test")


def create_yolo_dirs(reset: bool = False) -> None:
    if reset and YOLO_READY_DIR.exists():
        shutil.rmtree(YOLO_READY_DIR)
    for split in SPLITS:
        (YOLO_READY_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_READY_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)


def normalize_polygon(segmentation: list, width: float, height: float) -> list[str]:
    points = []
    for index in range(0, len(segmentation) - 1, 2):
        x = max(0.0, min(1.0, float(segmentation[index]) / width))
        y = max(0.0, min(1.0, float(segmentation[index + 1]) / height))
        points.extend((f"{x:.6f}", f"{y:.6f}"))
    return points


def normalize_bbox_xywh(bbox: list, width: float, height: float) -> list[str] | None:
    clamped = clamp_bbox_xywh(bbox, width, height)
    if clamped is None:
        return None
    x, y, w, h = clamped
    x_center = (x + (w / 2.0)) / width
    y_center = (y + (h / 2.0)) / height
    norm_w = w / width
    norm_h = h / height
    values = [x_center, y_center, norm_w, norm_h]
    if norm_w <= 0.0 or norm_h <= 0.0:
        return None
    values = [max(0.0, min(1.0, value)) for value in values]
    return [f"{value:.6f}" for value in values]


def yolo_line_for_annotation(annotation: dict, width: float, height: float) -> str | None:
    if YOLO_TASK == "detect":
        bbox = annotation_bbox(annotation, width, height)
        points = normalize_bbox_xywh(bbox, width=width, height=height) if bbox is not None else None
        if points is None:
            return None
        return "0 " + " ".join(points)

    lines = []
    for segmentation in annotation.get("segmentation", []):
        if not isinstance(segmentation, list) or len(segmentation) < 6:
            continue
        points = normalize_polygon(segmentation, width=width, height=height)
        if len(points) >= 6:
            lines.append(f"{annotation['category_id']} " + " ".join(points))
    return "\n".join(lines) if lines else None


def write_split_files(split: str, file_names: list[str], dist_dir: Path, label_content_by_file: dict[str, str]) -> None:
    for file_name in file_names:
        old_image_path = dist_dir / file_name
        new_image_path = YOLO_READY_DIR / "images" / split / file_name
        label_path = YOLO_READY_DIR / "labels" / split / Path(file_name).with_suffix(".txt")

        new_image_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(old_image_path, new_image_path)
        label_path.write_text(label_content_by_file[file_name] + "\n", encoding="utf-8")


def build_yolo_ready_dataset() -> dict:
    create_yolo_dirs(reset=True)
    summary = {"task": YOLO_TASK}

    for dataset_name in DATASETS:
        dist_dir = NORMALIZED_DIR / dataset_name
        if not dist_dir.exists():
            raise FileNotFoundError(f"Prepared dataset directory '{dist_dir}' not found.")

        files = [path.name for path in dist_dir.iterdir() if path.is_file() and path.suffix.lower() != ".json"]
        file_set = set(files)
        annotation_files = [path for path in dist_dir.iterdir() if path.is_file() and path.suffix.lower() == ".json"]
        if not annotation_files:
            raise FileNotFoundError(f"No annotation JSON found in '{dist_dir}'.")

        annotations = load_json(annotation_files[0])
        annotations_by_image = {}
        for annotation in annotations["annotations"]:
            annotations_by_image.setdefault(annotation["image_id"], []).append(annotation)

        label_content_by_file = {}
        labeled_files = []
        skipped_without_annotations = 0
        skipped_without_valid_labels = 0
        total_instances = 0

        for image in annotations["images"]:
            file_name = image["file_name"]
            if file_name not in file_set:
                raise FileNotFoundError(
                    f"Image file '{file_name}' not found in '{dist_dir}'. Please ensure the dataset is prepared correctly."
                )

            image_id = image["id"]
            image_annotations = annotations_by_image.get(image_id, [])
            if len(image_annotations) == 0:
                print(f"Warning: No annotations found for image '{file_name}' (id: {image_id}). This image will be skipped.")
                skipped_without_annotations += 1
                continue

            width = float(image["width"])
            height = float(image["height"])
            label_lines = []
            for annotation in image_annotations:
                label_text = yolo_line_for_annotation(annotation, width=width, height=height)
                if label_text:
                    label_lines.extend(line for line in label_text.splitlines() if line.strip())

            if not label_lines:
                skipped_without_valid_labels += 1
                continue

            label_content_by_file[file_name] = "\n".join(label_lines)
            labeled_files.append(file_name)
            total_instances += len(label_lines)

        train_size = int(len(labeled_files) * TRAIN_RATE)
        val_size = int(len(labeled_files) * VAL_RATE)
        test_size = len(labeled_files) - train_size - val_size

        split_files = {
            "train": labeled_files[:train_size],
            "val": labeled_files[train_size:train_size + val_size],
            "test": labeled_files[train_size + val_size:],
        }
        for split, file_names in split_files.items():
            write_split_files(split, file_names, dist_dir, label_content_by_file)

        summary[dataset_name] = {
            "source_images": len(files),
            "labeled_images": len(labeled_files),
            "instances": total_instances,
            "skipped_without_annotations": skipped_without_annotations,
            "skipped_without_valid_labels": skipped_without_valid_labels,
            "train": train_size,
            "val": val_size,
            "test": test_size,
        }

    return summary


In [ ]:
for dataset_name in DATASETS:
    print(f"Preparing dataset: {dataset_name}")
    prepare_dataset(dataset_name=dataset_name)

print("All datasets have been prepared and are ready for merging.")

summary = build_yolo_ready_dataset()
print(json.dumps(summary, indent=2))
print("\nYOLO-ready dataset:", YOLO_READY_DIR)


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def load_training_categories(categories_json: Path = AQUATRASH_ANNOTATIONS_PATH) -> list[dict]:
    if YOLO_TASK == "detect":
        return DETECT_CATEGORIES
    categories = load_json(categories_json).get("categories", [])
    if not categories:
        raise ValueError(f"No categories found in '{categories_json}'.")
    return sorted(categories, key=lambda item: int(item["id"]))


def read_label_ids(label_paths: list[Path]) -> set[int]:
    label_ids = set()
    for label_path in label_paths:
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if parts:
                label_ids.add(int(float(parts[0])))
    return label_ids


def infer_label_id_mode(label_ids: set[int], category_ids: set[int], zero_based_ids: set[int]) -> str:
    if not label_ids:
        return "zero_based"
    if label_ids <= zero_based_ids and 0 in label_ids:
        return "zero_based"
    if label_ids <= category_ids:
        return "category_ids"
    if label_ids <= zero_based_ids:
        return "zero_based"
    raise ValueError(f"Label IDs {sorted(label_ids)} do not match category IDs or zero-based YOLO IDs.")


def normalize_label_files_for_yolo(yolo_ready_dir: Path, categories: list[dict], mode: str = "auto") -> dict:
    label_paths = [path for split in SPLITS for path in (yolo_ready_dir / "labels" / split).rglob("*.txt")]
    original_id_to_yolo_id = {int(category["id"]): index for index, category in enumerate(categories)}
    category_ids = set(original_id_to_yolo_id)
    zero_based_ids = set(range(len(categories)))
    label_ids = read_label_ids(label_paths)

    if mode == "auto":
        mode = infer_label_id_mode(label_ids, category_ids, zero_based_ids)
    if mode not in {"category_ids", "zero_based"}:
        raise ValueError("LABEL_ID_MODE must be 'auto', 'category_ids', or 'zero_based'.")

    rewritten = 0
    for label_path in label_paths:
        changed = False
        output_lines = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            raw_class_id = int(float(parts[0]))
            if mode == "category_ids":
                if raw_class_id not in original_id_to_yolo_id:
                    raise ValueError(f"Unknown category ID {raw_class_id} in '{label_path}'.")
                class_id = original_id_to_yolo_id[raw_class_id]
            else:
                class_id = raw_class_id
            if class_id not in zero_based_ids:
                raise ValueError(f"YOLO class ID {class_id} in '{label_path}' is outside 0..{len(categories) - 1}.")
            new_line = " ".join([str(class_id), *parts[1:]])
            output_lines.append(new_line)
            changed = changed or new_line != line.strip()
        if changed:
            label_path.write_text("\n".join(output_lines) + ("\n" if output_lines else ""), encoding="utf-8")
            rewritten += 1

    return {
        "label_id_mode": mode,
        "labels_seen": len(label_paths),
        "rewritten_label_files": rewritten,
        "original_label_ids": sorted(label_ids),
    }


def split_counts_for_training(yolo_ready_dir: Path) -> dict:
    counts = {}
    for split in SPLITS:
        image_dir = yolo_ready_dir / "images" / split
        label_dir = yolo_ready_dir / "labels" / split
        counts[split] = {
            "images": sum(1 for path in image_dir.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS),
            "labels": sum(1 for path in label_dir.rglob("*.txt") if path.is_file()),
        }
    return counts


def build_yolo_dataset_yaml(yolo_ready_dir: Path = YOLO_READY_DIR, categories_json: Path = AQUATRASH_ANNOTATIONS_PATH) -> tuple[Path, dict]:
    yolo_ready_dir = Path(yolo_ready_dir).resolve()
    categories = load_training_categories(categories_json)
    names = [str(category["name"]) for category in categories]

    for split in SPLITS:
        if not (yolo_ready_dir / "images" / split).exists():
            raise FileNotFoundError(f"Missing images/{split} in '{yolo_ready_dir}'.")
        if not (yolo_ready_dir / "labels" / split).exists():
            raise FileNotFoundError(f"Missing labels/{split} in '{yolo_ready_dir}'.")

    label_summary = normalize_label_files_for_yolo(yolo_ready_dir, categories, mode=LABEL_ID_MODE)
    dataset_yaml_path = Path(DATASET_YAML_PATH).resolve()
    yaml_lines = [
        f"path: {yolo_ready_dir.as_posix()}",
        "train: images/train",
        "val: images/val",
        "test: images/test",
        f"nc: {len(names)}",
        "names:",
    ]
    yaml_lines.extend(f"  {index}: {name}" for index, name in enumerate(names))
    dataset_yaml_path.write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")

    summary = {
        "task": YOLO_TASK,
        "dataset_yaml": str(dataset_yaml_path),
        "yolo_ready_dir": str(yolo_ready_dir),
        "category_source": "one-class-trash" if YOLO_TASK == "detect" else str(Path(categories_json).resolve()),
        "classes": names,
        "splits": split_counts_for_training(yolo_ready_dir),
        **label_summary,
    }
    summary_path = yolo_ready_dir / "dataset_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return dataset_yaml_path, summary


def normalize_mlflow_uri(value: str | Path) -> str:
    text = str(value).strip()
    if "://" in text:
        return text
    return Path(text).expanduser().resolve().as_uri()


def setup_mlflow_tracking() -> None:
    tracking_uri = os.getenv("MLFLOW_TRACKING_URI") or str(MLRUNS_DIR)
    registry_uri = os.getenv("MLFLOW_REGISTRY_URI") or tracking_uri
    experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME") or EXPERIMENT_NAME
    mlflow.set_tracking_uri(normalize_mlflow_uri(tracking_uri))
    mlflow.set_registry_uri(normalize_mlflow_uri(registry_uri))
    mlflow.set_experiment(experiment_name)


def metric_results(metrics) -> dict:
    results = getattr(metrics, "results_dict", None)
    if not isinstance(results, dict):
        return {}
    clean = {}
    for key, value in results.items():
        try:
            clean[str(key)] = float(value)
        except (TypeError, ValueError):
            pass
    return clean


def metric_value(metrics, preferred: str) -> float:
    results = metric_results(metrics)
    aliases = {
        "mask_map50_95": ["metrics/mAP50-95(M)", "metrics/mAP50-95(Mask)", "metrics/mAP50-95(B)"],
        "mask_map50": ["metrics/mAP50(M)", "metrics/mAP50(Mask)", "metrics/mAP50(B)"],
        "box_map50_95": ["metrics/mAP50-95(B)", "metrics/mAP50-95(M)"],
        "box_map50": ["metrics/mAP50(B)", "metrics/mAP50(M)"],
    }
    for key in aliases.get(preferred, [preferred]):
        if key in results:
            return results[key]
    for key, value in results.items():
        if preferred.lower() in key.lower():
            return value
    return 0.0


def safe_metric_name(name: str) -> str:
    return name.replace("/", "_").replace("(", "").replace(")", "").replace(" ", "_")


def log_metric_dict(prefix: str, metrics) -> None:
    for key, value in metric_results(metrics).items():
        mlflow.log_metric(f"{prefix}_{safe_metric_name(key)}", value)


def yolo_train_kwargs(dataset_yaml_path: Path, run_name: str, epochs: int, overrides: dict) -> dict:
    kwargs = {
        "data": str(dataset_yaml_path),
        "epochs": epochs,
        "batch": BATCH,
        "imgsz": IMGSZ,
        "patience": PATIENCE,
        "device": DEVICE,
        "project": str(RUNS_DIR),
        "name": run_name,
        "exist_ok": True,
        "augment": AUGMENT,
        "mosaic": MOSAIC,
        "mixup": MIXUP,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "lrf": LRF,
        "weight_decay": WEIGHT_DECAY,
    }
    if YOLO_TASK == "seg":
        kwargs["copy_paste"] = COPY_PASTE
    kwargs.update(overrides)
    return kwargs


def train_yolo_ready_with_mlflow_optuna(dataset_yaml_path: Path, dataset_summary: dict) -> dict:
    global mlflow, optuna, YOLO
    import mlflow
    import optuna
    from ultralytics import YOLO

    setup_mlflow_tracking()
    best_params = {}

    with mlflow.start_run(run_name=RUN_NAME) as parent_run:
        mlflow.log_params({
            "task": YOLO_TASK,
            "model": YOLO_MODEL,
            "dataset_yaml": str(dataset_yaml_path),
            "epochs": EPOCHS,
            "batch": BATCH,
            "imgsz": IMGSZ,
            "optuna_trials": 0 if SKIP_OPTUNA else OPTUNA_TRIALS,
            "tune_epochs": TUNE_EPOCHS,
            "optuna_metric": OPTUNA_METRIC,
        })
        mlflow.log_artifact(str(dataset_yaml_path))
        summary_path = YOLO_READY_DIR / "dataset_summary.json"
        if summary_path.exists():
            mlflow.log_artifact(str(summary_path))

        if not SKIP_OPTUNA and OPTUNA_TRIALS > 0:
            def objective(trial: optuna.Trial) -> float:
                trial_params = {
                    "lr0": trial.suggest_float("lr0", 1e-4, 5e-2, log=True),
                    "lrf": trial.suggest_float("lrf", 0.01, 0.5, log=True),
                    "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
                    "mosaic": trial.suggest_float("mosaic", 0.0, 1.0),
                    "mixup": trial.suggest_float("mixup", 0.0, 0.3),
                }
                if YOLO_TASK == "seg":
                    trial_params["copy_paste"] = trial.suggest_float("copy_paste", 0.0, 0.5)
                trial_run_name = f"{RUN_NAME}-trial-{trial.number:03d}"
                with mlflow.start_run(run_name=trial_run_name, nested=True):
                    mlflow.log_params(trial_params)
                    model = YOLO(YOLO_MODEL)
                    model.train(**yolo_train_kwargs(dataset_yaml_path, trial_run_name, TUNE_EPOCHS, trial_params))
                    val_metrics = model.val(data=str(dataset_yaml_path), split="val", imgsz=IMGSZ)
                    score = metric_value(val_metrics, OPTUNA_METRIC)
                    mlflow.log_metric(OPTUNA_METRIC, score)
                    log_metric_dict("trial_val", val_metrics)
                    return score

            study = optuna.create_study(direction=OPTUNA_DIRECTION)
            study.optimize(objective, n_trials=OPTUNA_TRIALS)
            best_params = dict(study.best_params)
            best_params_path = RUNS_DIR / RUN_NAME / "optuna_best_params.json"
            best_params_path.parent.mkdir(parents=True, exist_ok=True)
            best_params_path.write_text(json.dumps(best_params, indent=2), encoding="utf-8")
            mlflow.log_params({f"best_{key}": value for key, value in best_params.items()})
            mlflow.log_metric(f"best_{OPTUNA_METRIC}", float(study.best_value))
            mlflow.log_artifact(str(best_params_path))

        final_model = YOLO(YOLO_MODEL)
        train_result = final_model.train(**yolo_train_kwargs(dataset_yaml_path, RUN_NAME, EPOCHS, best_params))
        run_dir = Path(getattr(train_result, "save_dir", None) or (RUNS_DIR / RUN_NAME))
        best_weights = run_dir / "weights" / "best.pt"
        if not best_weights.exists():
            best_weights = run_dir / "weights" / "last.pt"
        if not best_weights.exists():
            raise FileNotFoundError(f"Could not find trained weights under '{run_dir / 'weights'}'.")

        best_model = YOLO(str(best_weights))
        val_metrics = best_model.val(data=str(dataset_yaml_path), split="val", imgsz=IMGSZ)
        test_metrics = best_model.val(data=str(dataset_yaml_path), split="test", imgsz=IMGSZ)
        mlflow.log_metric("val_" + OPTUNA_METRIC, metric_value(val_metrics, OPTUNA_METRIC))
        mlflow.log_metric("test_" + OPTUNA_METRIC, metric_value(test_metrics, OPTUNA_METRIC))
        log_metric_dict("val", val_metrics)
        log_metric_dict("test", test_metrics)

        ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
        copied_weights = ARTIFACTS_DIR / f"{RUN_NAME}_best.pt"
        shutil.copy2(best_weights, copied_weights)
        mlflow.log_artifact(str(best_weights), artifact_path="weights")
        mlflow.log_artifact(str(copied_weights), artifact_path="exports")

        training_summary = {
            "parent_run_id": parent_run.info.run_id,
            "task": YOLO_TASK,
            "model": YOLO_MODEL,
            "dataset_yaml": str(dataset_yaml_path),
            "run_dir": str(run_dir),
            "best_weights": str(best_weights),
            "exported_weights": str(copied_weights),
            "best_params": best_params,
            "val_metrics": metric_results(val_metrics),
            "test_metrics": metric_results(test_metrics),
            "val_metric": metric_value(val_metrics, OPTUNA_METRIC),
            "test_metric": metric_value(test_metrics, OPTUNA_METRIC),
        }
        summary_output_path = run_dir / "training_summary.json"
        summary_output_path.write_text(json.dumps(training_summary, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(summary_output_path))
        return training_summary


In [ ]:
dataset_yaml_path, yolo_dataset_summary = build_yolo_dataset_yaml()
print(json.dumps(yolo_dataset_summary, indent=2))

if RUN_TRAINING:
    training_summary = train_yolo_ready_with_mlflow_optuna(dataset_yaml_path, yolo_dataset_summary)
    print(json.dumps(training_summary, indent=2))
else:
    print("RUN_TRAINING is False; dataset YAML was created but training was skipped.")


In [ ]:
def visualize_samples(samples: int, image_dir: Path, label_dir: Path, with_annotations: bool = True) -> None:
    if samples <= 0:
        return

    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
    import matplotlib.patches as patches
    from matplotlib.patches import Polygon

    image_dir = Path(image_dir)
    label_dir = Path(label_dir)
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = [path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in image_extensions]
    image_paths = random.sample(image_paths, min(samples, len(image_paths)))

    if not image_paths:
        print(f"No images found in '{image_dir}'.")
        return

    cols = min(3, len(image_paths))
    rows = (len(image_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)

    for ax in axes.flat:
        ax.axis("off")

    for ax, image_path in zip(axes.flat, image_paths):
        image = mpimg.imread(image_path)
        height, width = image.shape[:2]
        ax.imshow(image)
        ax.set_title(image_path.name, fontsize=9)

        if not with_annotations:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            continue

        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if YOLO_TASK == "detect":
                if len(parts) != 5:
                    continue
                try:
                    x_center, y_center, box_w, box_h = [float(value) for value in parts[1:5]]
                except ValueError:
                    continue
                x = (x_center - box_w / 2.0) * width
                y = (y_center - box_h / 2.0) * height
                rect_w = box_w * width
                rect_h = box_h * height
                ax.add_patch(patches.Rectangle((x, y), rect_w, rect_h, fill=False, edgecolor="yellow", linewidth=1.5))
                label_x, label_y = x, y
            else:
                if len(parts) < 7:
                    continue
                coord_count = (len(parts) - 1) // 2 * 2
                try:
                    coords = [float(value) for value in parts[1:1 + coord_count]]
                except ValueError:
                    continue
                points = [
                    (
                        max(0.0, min(1.0, coords[index])) * width,
                        max(0.0, min(1.0, coords[index + 1])) * height,
                    )
                    for index in range(0, len(coords), 2)
                ]
                if len(points) < 3:
                    continue
                ax.add_patch(Polygon(points, closed=True, fill=False, edgecolor="yellow", linewidth=1.5))
                label_x, label_y = points[0]

            ax.text(
                label_x,
                label_y,
                parts[0],
                color="black",
                fontsize=8,
                bbox={"facecolor": "yellow", "edgecolor": "none", "pad": 1},
            )

    plt.tight_layout()
    plt.show()


In [ ]:
if VISUALIZE_AFTER_BUILD:
    visualize_samples(
        samples=VISUALIZE_COUNT,
        image_dir=YOLO_READY_DIR / "images" / VISUALIZE_SPLIT,
        label_dir=YOLO_READY_DIR / "labels" / VISUALIZE_SPLIT,
        with_annotations=True,
    )